In [1]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import pandas as pd
from collections import Counter
import random
import ast

In [2]:
df = pd.read_csv("num_dataset_10.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [3]:
go_val = 10       #fed as the decoder's very first input only; never appears in the data itself
pad_val = 11
vocab_size = 12

In [4]:
inp_tensors = [torch.tensor(ast.literal_eval(s), dtype=torch.long) for s in df['source']]
out_tensors = [torch.tensor(ast.literal_eval(s), dtype=torch.long) for s in df['target']]

In [5]:
inp_padded = nn.utils.rnn.pad_sequence(inp_tensors, batch_first=True, padding_value=pad_val)
out_padded = nn.utils.rnn.pad_sequence(out_tensors, batch_first=True, padding_value=pad_val)

In [6]:
dataset = TensorDataset(inp_padded, out_padded)

#80, 10, 10 split for Train, Val, Test
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset,  batch_size=128, shuffle=False)

In [7]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size, num_layers):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        #Expands a token to dim embed_size, based on learned embeddings
        embedding = self.dropout(self.embedding(x))

        #outputs: hidden state at every timestep, shape (batch, seq_len, hidden_size)
        #         this is what attention will attend over
        #hidden:  final hidden state, shape (num_layers, batch, hidden_size)
        #         still used to initialize the decoder's hidden state
        outputs, hidden = self.gru(embedding)

        return outputs, hidden

In [8]:
class Attention(nn.Module):
    """Bahdanau-style additive attention.

    Scores each encoder timestep against the decoder's current hidden state,
    softmaxes those scores into weights, and returns a weighted sum of the
    encoder outputs (the context vector) plus the weights themselves (handy
    for visualizing what the model is attending to).
    """
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        #Projects [decoder_hidden ; encoder_output] -> a scalar-ish energy vector
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        # decoder_hidden:  (batch, hidden_size)          - last layer's hidden state
        # encoder_outputs: (batch, src_len, hidden_size)
        # mask:            (batch, src_len) bool, True where it's a real (non-pad) token
        src_len = encoder_outputs.shape[1]

        #Repeat decoder hidden state src_len times so it can be compared against
        #every encoder timestep: (batch, src_len, hidden_size)
        hidden_repeated = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)

        #Concatenate along the feature dim and squash through a linear + tanh
        energy = torch.tanh(self.attn(torch.cat((hidden_repeated, encoder_outputs), dim=2)))

        #Collapse hidden_size -> 1, giving one score per encoder timestep
        attention = self.v(energy).squeeze(2)  # (batch, src_len)

        #Prevent attending to <pad> positions
        if mask is not None:
            attention = attention.masked_fill(~mask, -1e10)

        weights = torch.softmax(attention, dim=1)  # (batch, src_len)

        #Weighted sum of encoder outputs -> context vector (batch, hidden_size)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context, weights

In [9]:
class Decoder(nn.Module):
    #Input and output sizes are the same here b/c both use same vocab space
    def __init__(self, input_size, embed_size, hidden_size, output_size, num_layers):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, embed_size)
        self.attention = Attention(hidden_size)

        #GRU input is now [embedded token ; attention context], hence embed_size + hidden_size
        self.gru = nn.GRU(embed_size + hidden_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)

        #fc takes [gru output ; context ; embedded token] -> vocab_size, giving it direct
        #access to the context vector rather than forcing everything through the GRU
        self.fc = nn.Linear(hidden_size * 2 + embed_size, output_size)

    def forward(self, x, hidden, encoder_outputs, mask=None):
        #Converts shape of x from (batch_size) to (batch_size, 1)
        x = x.unsqueeze(1)
        #Converts shape now to (batch_size, 1, embed_size)
        embedding = self.dropout(self.embedding(x))

        #Attend over encoder_outputs using the top layer's current hidden state
        context, attn_weights = self.attention(hidden[-1], encoder_outputs, mask)

        #Feed [embedded token ; context] into the GRU at this timestep
        gru_input = torch.cat((embedding, context.unsqueeze(1)), dim=2)
        outputs, hidden = self.gru(gru_input, hidden)

        #Squeeze the seq_len=1 dimension back out
        outputs = outputs.squeeze(1)
        embedding = embedding.squeeze(1)

        #Combine GRU output, context, and embedded input before the final projection
        predictions = self.fc(torch.cat((outputs, context, embedding), dim=1))

        return predictions, hidden, attn_weights

In [10]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_force_ratio):
        batch_size = source.shape[0]
        target_len = source.shape[1]

        #Tensor of zeros with shape (batch_size, target_len, vocab_size), will gradually be filled in by the decoder
        #BN: Vocab_size != Embedding_Size
        outputs = torch.zeros(batch_size, target_len, vocab_size).to(device)

        #Mask out <pad> positions in the source so attention can't look at them
        mask = (source != pad_val)

        #All encoder hidden states (for attention) + final hidden state (to seed the decoder)
        encoder_outputs, hidden = self.encoder(source)

        #<go> is not part of the source/target data - it only seeds the decoder's first input
        x = torch.full((batch_size,), go_val, dtype=torch.long).to(device)

        for t in range(target_len):
            output, hidden, _ = self.decoder(x, hidden, encoder_outputs, mask)
            #Fill in the outputs variable
            outputs[:, t] = output

            #Chooses the one with the highest probability
            best_guess = output.argmax(1)

            #Using random to ensure teacher forcing is applied only a certain percentage of the time
            if random.random() < teacher_force_ratio:
                x = target[:, t]
            else:
                x = best_guess

        return outputs

In [11]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, loader, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for src_batch, trg_batch in loader:
            src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
            output = model(src_batch, trg_batch, 0.0)
            output = output.reshape(-1, vocab_size)
            target = trg_batch.reshape(-1)
            loss = criterion(output, target)
            epoch_loss += loss.item()

        return epoch_loss / len(loader)

In [12]:
#Find the models accuracy on a Dataloader
def accuracy(model, loader, device):
    with torch.no_grad():
        model.eval()
        accurate = 0
        total = 0

        for src_batch, trg_batch in loader:
            src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
            output = model(src_batch, trg_batch, 0.0)

            output = output.reshape(-1, vocab_size)
            target = trg_batch.reshape(-1)

            #Creates list of booleans, False if 0, else True
            mask = target != pad_val
            preds = output.argmax(dim=1)

            #Ignoring padding tokens when evaluating
            accurate += (preds[mask] == target[mask]).sum().item()
            total += mask.sum().item()

        return accurate/total

In [13]:
def test(model, test_seq, return_attention=False):
    with torch.no_grad():
        model.eval()
        encoded = torch.tensor(test_seq, dtype=torch.long)

        #Pad our text to the length the model was trained on
        pad_len = inp_padded.shape[1] - len(encoded)
        encoded = torch.cat([encoded, torch.zeros(pad_len, dtype=torch.long).fill_(pad_val)])

        #Gives it batch size 1
        encoded = encoded.unsqueeze(0).to(device)
        mask = (encoded != pad_val)

        #All encoder hidden states (for attention) + final hidden state
        encoder_outputs, hidden = model.encoder(encoded)

        predicted = []
        attn_matrix = []
        #Seed the decoder with the <go> token - it's never part of the sequence itself
        decoder_input = torch.tensor([go_val], device=device)

        #No <eos> to signal a stop, so we rely on knowing the output length:
        #this is a copy task, so the output is exactly as long as the input
        for step in range(len(test_seq)):
            prediction, hidden, attn_weights = model.decoder(decoder_input, hidden, encoder_outputs, mask)
            best_guess = prediction.argmax(1).item()
            attn_matrix.append(attn_weights.squeeze(0).cpu().tolist())

            predicted.append(best_guess)
            decoder_input = torch.tensor([best_guess], device=device)

        if return_attention:
            return predicted, attn_matrix
        return predicted

In [16]:
#Training hyperparameters
num_epochs = 50
learning_rate = 1e-3
encoder_embedding_size = 64
decoder_embedding_size = 64
hidden_size = 128
num_layers = 1

#Model hyperparameters
encoder_net = Encoder(vocab_size, encoder_embedding_size, hidden_size, num_layers).to(device)
decoder_net = Decoder(vocab_size, decoder_embedding_size, hidden_size, vocab_size, num_layers).to(device)

model = Seq2Seq(encoder_net, decoder_net).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
#Ignore <pad>
criterion = nn.CrossEntropyLoss(ignore_index=pad_val)
#Combat overfitting
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
)

In [35]:
test_str = [8, 0, 3, 6, 5]
tf_ratio = 1
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
      for src_batch, trg_batch in train_loader:
          src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
          optimizer.zero_grad()

          output = model(src_batch, trg_batch, max(0.0, tf_ratio))

          #Flatten the batch size and seq length dimensions into 1
          output = output.reshape(-1, vocab_size)
          target = trg_batch.reshape(-1)

          loss = criterion(output, target)
          loss.backward()

          #Norm=magnitude, clipping gradient to ensure they don't explode
          torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1)
          optimizer.step()
          epoch_loss += loss.item()

    print(f"{test_str} => {test(model, test_str)}")
    train_loss = epoch_loss/len(train_loader)
    print(f"Train Loss: {train_loss}")
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Validation Loss: {val_loss}")
    tf_ratio -= 0.025
    scheduler.step(val_loss)

Epoch 1 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.273801622221712
Validation Loss: 0.0034853388344691623
Epoch 2 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.0059766644537448885
Validation Loss: 0.0006268472528077722
Epoch 3 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.0010278090400388463
Validation Loss: 0.0003549142002445897
Epoch 4 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.008732652490842156
Validation Loss: 0.00036249739868152473
Epoch 5 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.0007992220968008042
Validation Loss: 0.0003547864284233323
Epoch 6 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.0008132724503520876
Validation Loss: 0.0003382895374670625
Epoch 7 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.06937819694401696
Validation Loss: 0.0007090170069464616
Epoch 8 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.001112630207440816
Validation Loss: 0.0003520077046391273
Epoch 9 / 50
[8, 0, 3, 6, 5] => [5, 6, 

In [19]:
print(f"Train Loss: {evaluate(model, train_loader, criterion, device)}")
print(f"Train Acc: {accuracy(model, train_loader, device)}")
print(f"Val Loss: {evaluate(model, val_loader, criterion, device)}")
print(f"Val Acc: {accuracy(model, val_loader, device)}")
print(f"Test Loss: {evaluate(model, test_loader, criterion, device)}")
print(f"Test Acc: {accuracy(model, test_loader, device)}")

Train Loss: 0.00036967693961923945
Train Acc: 1.0
Val Loss: 0.0003700772016715731
Val Acc: 1.0
Test Loss: 0.00037007643480873887
Test Acc: 1.0


In [18]:
# torch.save(model.state_dict(), "no_sos_bos_attention_reversal.pt")
model.load_state_dict(torch.load("wo_bos_eos/no_sos_bos_attention_reversal.pt", weights_only=True, map_location=torch.device('cuda')))

<All keys matched successfully>

In [18]:
def build_context_df(model, loader, inp_padded, device):
    model.eval()
    all_sources = []
    all_targets = []
    all_contexts = []
    all_outputs = []

    with torch.no_grad():
        for src_batch, trg_batch in loader:
            src_batch = src_batch.to(device)
            trg_batch = trg_batch.to(device)
            encoder_outputs, hidden = model.encoder(src_batch)

            context = hidden[0]

            # Concatenate along feature dim: (batch_size, 2 * hidden_size)
            # To undo, h_final = context_vector[:, :128], c_final = context_vector[:, 128:]

            all_sources.append(src_batch.cpu())
            all_targets.append(trg_batch.cpu())
            all_contexts.append(context.cpu())
            all_outputs.append(encoder_outputs.cpu())

    all_sources = torch.cat(all_sources, dim=0)   # (N, seq_len)
    all_targets = torch.cat(all_targets, dim=0)   # (N, seq_len)
    all_contexts = torch.cat(all_contexts, dim=0)  # (N, 2 * hidden_size)
    all_outputs = torch.cat(all_outputs, dim=0)  # (N, 2 * hidden_size)

    source_lists = [row.tolist() for row in all_sources]
    target_lists = [row.tolist() for row in all_targets]
    context_lists = [row.tolist() for row in all_contexts]
    outputs_lists = [row.tolist() for row in all_outputs]

    context_df = pd.DataFrame({
        "source": source_lists,
        "context_vector": context_lists,
        "outputs" : outputs_lists,
        "target": target_lists
    })

    return context_df

full_loader = DataLoader(dataset, batch_size=32, shuffle=False)
context_df = build_context_df(model, full_loader, inp_padded, device)

In [19]:
def strip_padding(tokens):
    while tokens and tokens[-1] == pad_val:
        tokens = tokens[:-1]
    return tokens

context_df["source"] = context_df["source"].apply(strip_padding)
context_df["target"] = context_df["target"].apply(strip_padding)

In [20]:
context_df.head()

,source,context_vector,outputs,target
0,"[8, 1, 8, 2, 1, 6]","[0.5764219760894775, 0.8346996307373047, -0.82...","[[0.23297925293445587, 0.10470094531774521, -0...","[6, 1, 2, 8, 1, 8]"
1,"[2, 3, 3, 7, 9, 6]","[0.5781514644622803, 0.8351370692253113, -0.82...","[[0.21896734833717346, 0.11128649115562439, -0...","[6, 9, 7, 3, 3, 2]"
2,"[8, 5, 9, 4, 6, 7]","[0.5709876418113708, 0.831810474395752, -0.822...","[[0.23297925293445587, 0.10470094531774521, -0...","[7, 6, 4, 9, 5, 8]"
3,"[0, 6, 5, 9, 8, 7]","[0.57166588306427, 0.8306851983070374, -0.8210...","[[0.2669995129108429, 0.1201818585395813, -0.2...","[7, 8, 9, 5, 6, 0]"
4,"[2, 7, 3, 4, 4]","[0.535377562046051, 0.7399789094924927, -0.732...","[[0.21896734833717346, 0.11128649115562439, -0...","[4, 4, 3, 7, 2]"


In [21]:
source_parts = []
encoder_hiddens = []
for row_idx, row in context_df.iterrows():
    src_list = row['source']
    out_list = row['outputs']
 
    for i in range(len(src_list)):
        if (src_list[:i + 1] not in source_parts) and (len(src_list[:i + 1]) <= 6):
            source_parts.append(src_list[:i + 1])
            encoder_hiddens.append(out_list[i])
        
unfurled_df = pd.DataFrame({"source": source_parts, "output" : encoder_hiddens})
display(unfurled_df)

,source,output
0,[8],"[0.23297925293445587, 0.10470094531774521, -0...."
1,"[8, 1]","[0.34484654664993286, 0.29732629656791687, -0...."
2,"[8, 1, 8]","[0.4037744998931885, 0.5194594860076904, -0.61..."
3,"[8, 1, 8, 2]","[0.4335367679595947, 0.7069674730300903, -0.76..."
4,"[8, 1, 8, 2, 1]","[0.4994135797023773, 0.8279609680175781, -0.86..."
...,...,...
162579,"[3, 1, 8, 2, 4, 6, 2, 8]","[0.7620845437049866, 0.946745753288269, -0.952..."
162580,"[2, 7, 4, 1, 1]","[0.5041276812553406, 0.8298665881156921, -0.86..."
162581,"[2, 7, 4, 1, 1, 0]","[0.6209007501602173, 0.8992648124694824, -0.91..."
162582,"[2, 7, 4, 1, 1, 0, 0]","[0.6986156702041626, 0.9337235689163208, -0.94..."


In [22]:
unfurled_df.to_csv("no_sos_eos_attention_gru_reversal_data.csv", index=False)